## analysis R. static overview. each pair of regions, each category of clause.



### For two regions, r1 and r2, compare the clauses one by one; a regulation difference is counted whenever one is 1 and the other is 0.

one group as one long long long matrix

In [48]:
import pandas as pd
import numpy as np
from itertools import combinations
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.formatting.rule import ColorScaleRule
from openpyxl.utils import get_column_letter
from pathlib import Path

In [49]:
input_folder = Path("dataset\AA1_first_batch\AA2_second_100_batch\pp_matrix_output")
output_file = "dataset\\AA1_first_batch\AA2_second_100_batch\\analysis_pp_output\\privacy_policy_difference_rate_results.xlsx"

def compute_diff_rate_matrix_from_policy(policy_df, clause_col="Clause"):
    mat = policy_df.set_index(clause_col)

    regions = list(mat.columns)
    num_clauses = mat.shape[0]

    mat = mat.astype(int)

    diff_rate_matrix = pd.DataFrame(
        np.zeros((len(regions), len(regions))),
        index=regions,
        columns=regions
    )

    for r1, r2 in combinations(regions, 2):
        diff_count = (mat[r1] != mat[r2]).sum()
        diff_rate = diff_count / num_clauses

        diff_rate_matrix.loc[r1, r2] = diff_rate
        diff_rate_matrix.loc[r2, r1] = diff_rate

    return diff_rate_matrix


app_matrices = {}

for file_path in input_folder.glob("*.csv"):
    app_name = file_path.stem

    policy_df = pd.read_csv(file_path)

    diff_rate_matrix = compute_diff_rate_matrix_from_policy(policy_df)
    print(f"Diff rate matrix for {app_name}:\n{diff_rate_matrix}")

    app_matrices[app_name] = diff_rate_matrix

print("Number of apps:", len(app_matrices))

# average matrix
first_matrix = next(iter(app_matrices.values()))
avg_matrix = pd.DataFrame(
    np.zeros(first_matrix.shape),
    index=first_matrix.index,
    columns=first_matrix.columns
)

for app_name, matrix in app_matrices.items():
    avg_matrix += matrix

avg_matrix = avg_matrix / len(app_matrices)

# save
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    avg_matrix.to_excel(writer, sheet_name="average_diff_rate")

    for app_name, matrix in app_matrices.items():
        sheet_name = str(app_name)[:31]
        matrix.to_excel(writer, sheet_name=sheet_name)

print(f"Saved to {output_file}")

Diff rate matrix for com.brainium.solitairefree:
            California     India     Texas   Vietnam  Bangladesh     Egypt  \
California    0.000000  0.000000  0.000000  0.000000    0.019608  0.019608   
India         0.000000  0.000000  0.000000  0.000000    0.019608  0.019608   
Texas         0.000000  0.000000  0.000000  0.000000    0.019608  0.019608   
Vietnam       0.000000  0.000000  0.000000  0.000000    0.019608  0.019608   
Bangladesh    0.019608  0.019608  0.019608  0.019608    0.000000  0.000000   
Egypt         0.019608  0.019608  0.019608  0.019608    0.000000  0.000000   
Germany       0.019608  0.019608  0.019608  0.019608    0.000000  0.000000   
Nigeria       0.019608  0.019608  0.019608  0.019608    0.000000  0.000000   
Pakistan      0.019608  0.019608  0.019608  0.019608    0.000000  0.000000   
Saudi         0.019608  0.019608  0.019608  0.019608    0.000000  0.000000   
Turkey        0.019608  0.019608  0.019608  0.019608    0.000000  0.000000   

             G

### Among clauses disclosed by at least one of the two regions, what fraction are disclosed differently?

In [50]:
import pandas as pd
import numpy as np
from itertools import combinations
from pathlib import Path

# =========================
# Settings
# =========================

input_folder = Path(r"dataset\AA1_first_batch\AA2_second_100_batch\pp_matrix_output")
output_file = "dataset\\AA1_first_batch\AA2_second_100_batch\\analysis_pp_output\\group_privacy_policy_pairwise_active_diff_rate.xlsx"

CLAUSE_COL = "Clause"


# =========================
# Function:
# Compute one app's pairwise active difference rate matrix
# =========================

def compute_pairwise_active_diff_rate_matrix(policy_df, clause_col="Clause"):
    """
    Input format:
        rows = clauses
        columns = Clause + regions
        values = 0/1

    For each region pair:
        denominator = clauses disclosed by at least one of the two regions
        numerator = clauses whose disclosure values differ between the two regions

    Difference rate:
        diff_rate = numerator / denominator

    If both regions disclose nothing for all clauses, denominator = 0, return NaN.
    """

    # Remove unnamed index columns if they exist
    policy_df = policy_df.loc[:, ~policy_df.columns.str.contains("^Unnamed")]

    if clause_col not in policy_df.columns:
        raise ValueError(f"Cannot find clause column: {clause_col}")

    # Convert to clause x region matrix
    mat = policy_df.set_index(clause_col)

    # Remove fully empty columns
    mat = mat.dropna(axis=1, how="all")

    # Missing value means not disclosed
    mat = mat.fillna(0)

    # Convert to integer 0/1
    mat = mat.astype(int)

    regions = list(mat.columns)

    D = pd.DataFrame(
        np.zeros((len(regions), len(regions))),
        index=regions,
        columns=regions
    )

    for r1, r2 in combinations(regions, 2):
        # clauses disclosed by at least one of the two regions
        active_clauses = (mat[r1] == 1) | (mat[r2] == 1)

        denominator = active_clauses.sum()

        if denominator == 0:
            diff_rate = np.nan
        else:
            diff_count = (
                mat.loc[active_clauses, r1] != mat.loc[active_clauses, r2]
            ).sum()

            diff_rate = diff_count / denominator

        D.loc[r1, r2] = diff_rate
        D.loc[r2, r1] = diff_rate
    print(f"Pairwise active difference rate matrix for {app_name}:\n{D}\n")
    return D


# =========================
# Function:
# Average matrices, ignoring NaN
# =========================

def average_matrices_ignore_nan(app_matrices):
    """
    Element-wise average over app-level matrices.
    NaN values are ignored.
    """

    matrices = list(app_matrices.values())

    if len(matrices) == 0:
        raise ValueError("No app matrices found.")

    stacked = np.stack([m.values for m in matrices], axis=0)

    avg_values = np.nanmean(stacked, axis=0)

    first_matrix = matrices[0]

    avg_matrix = pd.DataFrame(
        avg_values,
        index=first_matrix.index,
        columns=first_matrix.columns
    )

    return avg_matrix


# =========================
# Main:
# Read all CSV files and compute app-level matrices
# =========================

app_matrices = {}
summary_records = []

for file_path in input_folder.glob("*.csv"):
    app_name = file_path.stem

    policy_df = pd.read_csv(file_path)

    if policy_df.shape[1] != 12:
        print(f"Warning: {app_name} has {policy_df.shape[1]} regions, expected 12.")
        continue

    diff_rate_matrix = compute_pairwise_active_diff_rate_matrix(
        policy_df,
        clause_col=CLAUSE_COL
    )

    app_matrices[app_name] = diff_rate_matrix

    # upper triangle values, excluding diagonal
    upper_values = diff_rate_matrix.values[
        np.triu_indices_from(diff_rate_matrix.values, k=1)
    ]

    valid_values = upper_values[~np.isnan(upper_values)]

    summary_records.append({
        "app": app_name,
        "num_region_pairs": len(upper_values),
        "num_valid_region_pairs": len(valid_values),
        "mean_pairwise_active_diff_rate": np.nanmean(valid_values) if len(valid_values) > 0 else np.nan,
        "max_pairwise_active_diff_rate": np.nanmax(valid_values) if len(valid_values) > 0 else np.nan,
        "has_any_policy_region_variation": np.nanmax(valid_values) > 0 if len(valid_values) > 0 else False
    })

    print(f"Diff rate matrix for {app_name}:\n{diff_rate_matrix}\n")

print("Number of apps:", len(app_matrices))


# =========================
# Compute group-level average matrix
# =========================

avg_matrix = average_matrices_ignore_nan(app_matrices)

summary_df = pd.DataFrame(summary_records)


# =========================
# Flatten group-level matrix into pairwise table
# =========================

avg_pairwise_records = []

regions = list(avg_matrix.index)

for r1, r2 in combinations(regions, 2):
    avg_pairwise_records.append({
        "region_1": r1,
        "region_2": r2,
        "avg_pairwise_active_policy_diff_rate": avg_matrix.loc[r1, r2]
    })

avg_pairwise_df = pd.DataFrame(avg_pairwise_records)


# =========================
# Save outputs
# =========================

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    # group-level matrix
    avg_matrix.to_excel(writer, sheet_name="avg_region_matrix")

    # group-level pairwise table
    avg_pairwise_df.to_excel(writer, sheet_name="avg_pairwise_table", index=False)

    # app-level summary
    summary_df.to_excel(writer, sheet_name="app_summary", index=False)

    # app-level matrices
    for app_name, matrix in app_matrices.items():
        sheet_name = str(app_name)[:31]
        matrix.to_excel(writer, sheet_name=sheet_name)

print(f"Saved results to {output_file}")

print("Apps with any policy regional variation:",
      summary_df["has_any_policy_region_variation"].sum())

print("Apps with no policy regional variation:",
      (~summary_df["has_any_policy_region_variation"]).sum())

print("Group-level average privacy policy region-region matrix:")
print(avg_matrix)

Pairwise active difference rate matrix for com.brainium.solitairefree:
            California     India     Texas   Vietnam  Bangladesh     Egypt  \
California    0.000000  0.000000  0.000000  0.000000    0.333333  0.333333   
India         0.000000  0.000000  0.000000  0.000000    0.333333  0.333333   
Texas         0.000000  0.000000  0.000000  0.000000    0.333333  0.333333   
Vietnam       0.000000  0.000000  0.000000  0.000000    0.333333  0.333333   
Bangladesh    0.333333  0.333333  0.333333  0.333333    0.000000  0.000000   
Egypt         0.333333  0.333333  0.333333  0.333333    0.000000  0.000000   
Germany       0.333333  0.333333  0.333333  0.333333    0.000000  0.000000   
Nigeria       0.333333  0.333333  0.333333  0.333333    0.000000  0.000000   
Pakistan      0.333333  0.333333  0.333333  0.333333    0.000000  0.000000   
Saudi         0.333333  0.333333  0.333333  0.333333    0.000000  0.000000   
Turkey        0.333333  0.333333  0.333333  0.333333    0.000000  0.000

### result

In [52]:
import pandas as pd
import numpy as np
from pathlib import Path
from itertools import combinations

# =========================
# Settings
# =========================

input_folder = Path(r"dataset\AA1_first_batch\AA2_second_100_batch\pp_matrix_output")
output_file = "dataset\\AA1_first_batch\AA2_second_100_batch\\analysis_pp_output\\privacy_policy_pairwise_active_diff_analysis.xlsx"


CLAUSE_COL = "Clause"


# =========================
# Clause category mapping
# =========================
# If your clauses are named like P1, P2, CR1, C1, R1, you can use this function directly.
# If your category cannot be determined by prefix, modify it manually here.

def get_clause_category(clause):
    clause = str(clause).strip()

    if clause.startswith("CR"):
        return "Controller & Recipient"
    elif clause.startswith("P"):
        return "Personal Data Practices"
    elif clause.startswith("C"):
        return "Consent"
    elif clause.startswith("R"):
        return "Rights"
    else:
        return "Others"


# =========================
# Load one app matrix
# =========================

def load_policy_matrix(policy_df, clause_col=CLAUSE_COL):
    """
    Input format:
        rows = clauses
        columns = Clause + regions
        values = 0/1

    Output:
        mat: clause x region matrix
    """

    # remove unnamed columns
    policy_df = policy_df.loc[:, ~policy_df.columns.str.contains("^Unnamed")]

    if clause_col not in policy_df.columns:
        raise ValueError(f"Cannot find clause column: {clause_col}")

    mat = policy_df.set_index(clause_col)

    # remove empty columns
    mat = mat.dropna(axis=1, how="all")

    # missing means not disclosed
    mat = mat.fillna(0)

    # convert to integer
    mat = mat.astype(int)

    return mat


# =========================
# Analyze one app
# =========================

def analyze_one_app(mat, app_name):
    """
    For one app, compute:
    1. pairwise active difference rate matrix
    2. pair-region-clause-level records
    3. pair-region-category-level records

    Pairwise active difference rate:
        denominator = clauses disclosed by at least one of the two regions
        numerator = clauses whose disclosure values differ between two regions
    """

    regions = list(mat.columns)
    clauses = list(mat.index)

    # region x region matrix
    D = pd.DataFrame(
        np.zeros((len(regions), len(regions))),
        index=regions,
        columns=regions
    )

    pair_records = []
    category_records = []
    clause_records = []

    for r1, r2 in combinations(regions, 2):

        # active clauses for this region pair
        active_mask = (mat[r1] == 1) | (mat[r2] == 1)
        active_clauses = mat.index[active_mask]

        denominator = len(active_clauses)

        if denominator == 0:
            diff_rate = np.nan
            diff_count = 0
        else:
            diff_mask = mat.loc[active_clauses, r1] != mat.loc[active_clauses, r2]
            diff_clauses = active_clauses[diff_mask]

            diff_count = len(diff_clauses)
            diff_rate = diff_count / denominator

        D.loc[r1, r2] = diff_rate
        D.loc[r2, r1] = diff_rate

        pair_records.append({
            "app": app_name,
            "region_1": r1,
            "region_2": r2,
            "active_clause_count": denominator,
            "diff_clause_count": diff_count,
            "pairwise_active_diff_rate": diff_rate
        })

        # clause-level contribution
        # For each clause, record whether this region pair differs and its contribution to the pairwise active diff rate
        for clause in clauses:
            category = get_clause_category(clause)

            is_active = int((mat.loc[clause, r1] == 1) or (mat.loc[clause, r2] == 1))
            is_diff = int(mat.loc[clause, r1] != mat.loc[clause, r2])

            if denominator == 0:
                contribution_to_diff_rate = np.nan
            else:
                # Only active clauses can contribute to the pairwise active diff rate
                contribution_to_diff_rate = is_diff / denominator if is_active else 0

            clause_records.append({
                "app": app_name,
                "region_1": r1,
                "region_2": r2,
                "clause": clause,
                "category": category,
                "region_1_value": mat.loc[clause, r1],
                "region_2_value": mat.loc[clause, r2],
                "is_active_for_pair": is_active,
                "is_diff": is_diff,
                "contribution_to_pairwise_active_diff_rate": contribution_to_diff_rate
            })

        # category-level contribution
        if denominator == 0:
            # no active clauses
            categories = sorted(set(get_clause_category(c) for c in clauses))
            for category in categories:
                category_records.append({
                    "app": app_name,
                    "region_1": r1,
                    "region_2": r2,
                    "category": category,
                    "category_active_clause_count": 0,
                    "category_diff_clause_count": 0,
                    "category_contribution_to_diff_rate": np.nan,
                    "category_share_among_diff_clauses": np.nan
                })
        else:
            # build temp df for active clauses
            temp = []

            for clause in active_clauses:
                category = get_clause_category(clause)
                is_diff = int(mat.loc[clause, r1] != mat.loc[clause, r2])

                temp.append({
                    "clause": clause,
                    "category": category,
                    "is_diff": is_diff
                })

            temp_df = pd.DataFrame(temp)

            for category, sub in temp_df.groupby("category"):
                category_active_count = len(sub)
                category_diff_count = sub["is_diff"].sum()

                # Contribution of this category to the final pairwise active diff rate
                # The sum of contributions from all categories equals pairwise_active_diff_rate
                category_contribution = category_diff_count / denominator

                # Share of this category among all diff clauses
                if diff_count == 0:
                    category_share = 0
                else:
                    category_share = category_diff_count / diff_count

                category_records.append({
                    "app": app_name,
                    "region_1": r1,
                    "region_2": r2,
                    "category": category,
                    "category_active_clause_count": category_active_count,
                    "category_diff_clause_count": category_diff_count,
                    "category_contribution_to_diff_rate": category_contribution,
                    "category_share_among_diff_clauses": category_share
                })

    return D, pair_records, category_records, clause_records


# =========================
# Main
# =========================

app_matrices = {}
all_pair_records = []
all_category_records = []
all_clause_records = []

for file_path in input_folder.glob("*.csv"):
    app_name = file_path.stem

    policy_df = pd.read_csv(file_path)

    if policy_df.shape[1] != 12:
        print(f"Warning: {app_name} has {policy_df.shape[1]} regions, expected 12.")
        continue


    mat = load_policy_matrix(policy_df)

    D, pair_records, category_records, clause_records = analyze_one_app(mat, app_name)

    app_matrices[app_name] = D
    all_pair_records.extend(pair_records)
    all_category_records.extend(category_records)
    all_clause_records.extend(clause_records)

print("Number of apps:", len(app_matrices))


# =========================
# Convert records to DataFrames
# =========================

pair_df = pd.DataFrame(all_pair_records)
category_df = pd.DataFrame(all_category_records)
clause_df = pd.DataFrame(all_clause_records)


# =========================
# Group-level average region x region matrix
# =========================

def average_matrices_ignore_nan(app_matrices):
    matrices = list(app_matrices.values())

    stacked = np.stack([m.values for m in matrices], axis=0)
    avg_values = np.nanmean(stacked, axis=0)

    first = matrices[0]

    avg_matrix = pd.DataFrame(
        avg_values,
        index=first.index,
        columns=first.columns
    )

    return avg_matrix

avg_matrix = average_matrices_ignore_nan(app_matrices)

path_avg_matrix = "dataset\\AA1_first_batch\\AA2_second_100_batch\\analysis_pp_output\\PP_overall_matrix.csv" 
avg_matrix.to_csv(path_avg_matrix, index=True)


# =========================
# 1. Which region pairs have largest differences?
# =========================

avg_pairwise_df = (
    pair_df
    .groupby(["region_1", "region_2"], as_index=False)
    .agg(
        avg_pairwise_active_diff_rate=("pairwise_active_diff_rate", "mean"),
        avg_active_clause_count=("active_clause_count", "mean"),
        avg_diff_clause_count=("diff_clause_count", "mean"),
        num_apps=("app", "nunique")
    )
    .sort_values("avg_pairwise_active_diff_rate", ascending=False)
)

print("\nTop region pairs by average pairwise active diff rate:")
print(avg_pairwise_df.head(20))


# =========================
# 2. For each region pair, which categories contribute most?
# =========================

pair_category_contribution_df = (
    category_df
    .groupby(["region_1", "region_2", "category"], as_index=False)
    .agg(
        avg_category_contribution_to_diff_rate=("category_contribution_to_diff_rate", "mean"),
        avg_category_share_among_diff_clauses=("category_share_among_diff_clauses", "mean"),
        avg_category_active_clause_count=("category_active_clause_count", "mean"),
        avg_category_diff_clause_count=("category_diff_clause_count", "mean"),
        num_apps=("app", "nunique")
    )
    .sort_values(
        ["region_1", "region_2", "avg_category_contribution_to_diff_rate"],
        ascending=[True, True, False]
    )
)

# top category per region pair
top_category_per_pair_df = (
    pair_category_contribution_df
    .sort_values(
        ["region_1", "region_2", "avg_category_contribution_to_diff_rate"],
        ascending=[True, True, False]
    )
    .groupby(["region_1", "region_2"], as_index=False)
    .head(3)
)

print("\nTop categories for each region pair:")
print(top_category_per_pair_df.head(30))


# =========================
# 3. For each region pair, which clauses contribute most?
# =========================

pair_clause_contribution_df = (
    clause_df
    .groupby(["region_1", "region_2", "clause", "category"], as_index=False)
    .agg(
        avg_clause_contribution_to_diff_rate=("contribution_to_pairwise_active_diff_rate", "mean"),
        diff_frequency_across_apps=("is_diff", "mean"),
        active_frequency_across_apps=("is_active_for_pair", "mean"),
        num_apps=("app", "nunique")
    )
    .sort_values(
        ["region_1", "region_2", "avg_clause_contribution_to_diff_rate"],
        ascending=[True, True, False]
    )
)

# top clauses per region pair
top_clause_per_pair_df = (
    pair_clause_contribution_df
    .sort_values(
        ["region_1", "region_2", "avg_clause_contribution_to_diff_rate"],
        ascending=[True, True, False]
    )
    .groupby(["region_1", "region_2"], as_index=False)
    .head(10)
)

print("\nTop clauses for each region pair:")
print(top_clause_per_pair_df.head(30))


# =========================
# 4. From each clause's perspective: clause difference ranking
# =========================

clause_overall_ranking_df = (
    clause_df
    .groupby(["clause", "category"], as_index=False)
    .agg(
        avg_contribution_to_pairwise_active_diff_rate=("contribution_to_pairwise_active_diff_rate", "mean"),
        diff_frequency_across_all_app_region_pairs=("is_diff", "mean"),
        active_frequency_across_all_app_region_pairs=("is_active_for_pair", "mean"),
        num_records=("app", "count")
    )
    .sort_values("avg_contribution_to_pairwise_active_diff_rate", ascending=False)
)

print("\nClause overall difference ranking:")
print(clause_overall_ranking_df.head(30))


# =========================
# 5. Category overall ranking
# =========================

category_overall_ranking_df = (
    category_df
    .groupby("category", as_index=False)
    .agg(
        avg_category_contribution_to_diff_rate=("category_contribution_to_diff_rate", "mean"),
        avg_category_share_among_diff_clauses=("category_share_among_diff_clauses", "mean"),
        avg_category_active_clause_count=("category_active_clause_count", "mean"),
        avg_category_diff_clause_count=("category_diff_clause_count", "mean")
    )
    .sort_values("avg_category_contribution_to_diff_rate", ascending=False)
)

print("\nCategory overall ranking:")
print(category_overall_ranking_df)


# =========================
# 6. Clause x region-pair matrix
# =========================
# This table shows the region pairs in which each clause has the largest differences.
# value = fraction of apps where this clause differs for that region pair.

clause_region_pair_diff_df = (
    clause_df
    .groupby(["clause", "category", "region_1", "region_2"], as_index=False)
    .agg(
        diff_frequency_across_apps=("is_diff", "mean"),
        active_frequency_across_apps=("is_active_for_pair", "mean"),
        avg_contribution_to_pairwise_active_diff_rate=("contribution_to_pairwise_active_diff_rate", "mean")
    )
    .sort_values(
        ["clause", "diff_frequency_across_apps"],
        ascending=[True, False]
    )
)


# =========================
# 7. For top-N region pairs, show top clauses and categories
# =========================

top_n_pairs = 10

top_pairs = avg_pairwise_df.head(top_n_pairs)[["region_1", "region_2"]]

top_pair_keys = set(
    zip(top_pairs["region_1"], top_pairs["region_2"])
)

top_pairs_category_detail_df = pair_category_contribution_df[
    pair_category_contribution_df.apply(
        lambda row: (row["region_1"], row["region_2"]) in top_pair_keys,
        axis=1
    )
].sort_values(
    ["region_1", "region_2", "avg_category_contribution_to_diff_rate"],
    ascending=[True, True, False]
)

top_pairs_clause_detail_df = pair_clause_contribution_df[
    pair_clause_contribution_df.apply(
        lambda row: (row["region_1"], row["region_2"]) in top_pair_keys,
        axis=1
    )
].sort_values(
    ["region_1", "region_2", "avg_clause_contribution_to_diff_rate"],
    ascending=[True, True, False]
)


# =========================
# Save outputs
# =========================

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    # final group-level matrix
    avg_matrix.to_excel(writer, sheet_name="avg_region_matrix")

    # pair ranking
    avg_pairwise_df.to_excel(writer, sheet_name="region_pair_ranking", index=False)

    # category analysis
    pair_category_contribution_df.to_excel(writer, sheet_name="pair_category_all", index=False)
    top_category_per_pair_df.to_excel(writer, sheet_name="top_categories_per_pair", index=False)
    category_overall_ranking_df.to_excel(writer, sheet_name="category_overall_ranking", index=False)

    # clause analysis
    pair_clause_contribution_df.to_excel(writer, sheet_name="pair_clause_all", index=False)
    top_clause_per_pair_df.to_excel(writer, sheet_name="top_clauses_per_pair", index=False)
    clause_overall_ranking_df.to_excel(writer, sheet_name="clause_overall_ranking", index=False)
    clause_region_pair_diff_df.to_excel(writer, sheet_name="clause_region_pair_diff", index=False)

    # app summary and raw pair data
    pair_df.to_excel(writer, sheet_name="app_pair_raw", index=False)

    # details for top region pairs
    top_pairs_category_detail_df.to_excel(writer, sheet_name="top_pairs_category_detail", index=False)
    top_pairs_clause_detail_df.to_excel(writer, sheet_name="top_pairs_clause_detail", index=False)

print(f"\nSaved analysis to {output_file}")

Number of apps: 23

Top region pairs by average pairwise active diff rate:
    region_1    region_2  avg_pairwise_active_diff_rate  \
63  Pakistan     Nigeria                       1.000000   
61  Pakistan       Egypt                       0.714286   
60  Pakistan  California                       0.428571   
50   Nigeria  California                       0.428571   
30   Germany  California                       0.428571   
31   Germany       Egypt                       0.385714   
86    Turkey       India                       0.332143   
84    Turkey       Egypt                       0.327289   
89    Turkey       Saudi                       0.321769   
87    Turkey     Nigeria                       0.277183   
94   Vietnam       Egypt                       0.275510   
99   Vietnam       Saudi                       0.267857   
74     Texas       Egypt                       0.251166   
79     Texas       Saudi                       0.251082   
62  Pakistan     Germany                

## analysis regulation qualified pp differences.

### first, mapping the pp with regulation matrix, r=1,p=1, then 1; r=0,p=0, then 0; the leftover lable as np.nan.

In [53]:
from pathlib import Path
import pandas as pd
import numpy as np

# Folder containing input privacy-policy matrices
input_folder = Path(r"dataset\AA1_first_batch\AA2_second_100_batch\pp_matrix_output")

# Regulation matrix path; replace it with your own file
regulation_path = Path(r"dataset\regions_regulation.xlsx")

# Output folder
output_folder = Path(r"dataset\AA1_first_batch\AA2_second_100_batch\pp_regulation_matched_output")
output_folder.mkdir(parents=True, exist_ok=True)

# Set this to True if the first CSV column is a clause name / clause ID
# If all 12 columns are region columns and there is no clause-ID column, change this to False
HAS_CLAUSE_ID_COL = True

In [54]:
def binarize_matrix_values(df: pd.DataFrame, data_cols):
    """
    Convert data columns to 0/1.
    Also handle original values such as True/False and yes/no when possible.
    """
    tmp = df.copy()

    for col in data_cols:
        tmp[col] = (
            tmp[col]
            .replace({
                True: 1,
                False: 0,
                "True": 1,
                "False": 0,
                "true": 1,
                "false": 0,
                "YES": 1,
                "NO": 0,
                "Yes": 1,
                "No": 0,
                "yes": 1,
                "no": 0,
            })
        )
        tmp[col] = pd.to_numeric(tmp[col], errors="coerce")

    return tmp

In [55]:
# Read the regulation matrix
# If the target is not the first sheet in your XLSX file, change sheet_name
reg_df = pd.read_excel(regulation_path, sheet_name=0)
reg_df.head(), reg_df.shape

# # Remove Unnamed empty columns if present
# reg_df = reg_df.loc[:, ~reg_df.columns.astype(str).str.startswith("Unnamed")]

# Check the number of columns in the regulation matrix
if reg_df.shape[1] != 12:
    raise ValueError(f"Regulation matrix does not have 12 columns; it currently has {reg_df.shape[1]} columns")

# # The first column is the clause ID, followed by region columns
clause_col = reg_df.columns[0]
region_cols = list(reg_df.columns[1:])
clause_col, region_cols

# Set clause as the index to facilitate alignment with the PP matrix
reg_df = reg_df.set_index(clause_col)
reg_df.head()

# Convert the regulation region columns to numeric 0/1 values
# reg_df[region_cols] = reg_df[region_cols].apply(pd.to_numeric, errors="coerce")
# 1. Fill all NaN values with 0
# 2. Convert the entire data type to integer (int)
reg_df[region_cols] = reg_df[region_cols].fillna(0).astype(int)
reg_df.head()

,California,Texas,Germany,Turkey,Egypt,Vietnam,Nigeria,India,Saudi,Bangladesh,Pakistan
Clause,,,,,,,,,,,
P1,1,1,1,1,1,1,1,1,1,1,1
P2,1,1,0,0,0,0,1,0,1,0,0
P3,1,0,0,0,0,0,0,0,0,0,0
P4,1,1,1,1,1,1,1,1,1,1,1
P5,1,0,0,1,0,0,0,0,0,0,0


In [56]:
processed = 0
skipped = 0

for pp_path in input_folder.glob("*.csv"):
    try:
        pp_df = pd.read_csv(pp_path)

        # Remove possible empty Unnamed columns
        pp_df = pp_df.loc[:, ~pp_df.columns.astype(str).str.startswith("Unnamed")]

        # Skip if the input PP matrix does not have 12 columns
        if pp_df.shape[1] != 12:
            print(f"[SKIP] {pp_path.name}: columns = {pp_df.shape[1]}, not 12")
            skipped += 1
            continue

        pp_clause_col = pp_df.columns[0]
        pp_region_cols = list(pp_df.columns[1:])

        # # Check whether the region columns are consistent
        # if pp_region_cols != region_cols:
        #     print(f"[SKIP] {pp_path.name}: region columns not matched")
        #     print("PP columns :", pp_region_cols)
        #     print("REG columns:", region_cols)
        #     skipped += 1
        #     continue

        pp_df = pp_df.set_index(pp_clause_col)

        # Convert to numeric 0/1 values
        # pp_df[region_cols] = pp_df[region_cols].apply(pd.to_numeric, errors="coerce")
        pp_df[region_cols] = pp_df[region_cols].fillna(0).astype(int)
        print(f"[INFO] {pp_path.name}:\n", pp_df.head())

        # Align clauses
        common_clauses = reg_df.index.intersection(pp_df.index)

        if len(common_clauses) == 0:
            print(f"[SKIP] {pp_path.name}: no common clauses")
            skipped += 1
            continue

        reg_aligned = reg_df.loc[common_clauses, region_cols]
        pp_aligned = pp_df.loc[common_clauses, region_cols]

        # Core rule:
        # R=1, P=1 -> 1
        # R=0, P=0 -> 0
        # Other values -> np.nan
        result_values = np.where(
            (reg_aligned == 1) & (pp_aligned == 1),
            1,
            np.where(
                (reg_aligned == 0) & (pp_aligned == 0),
                0,
                np.nan
            )
        )

        result_df = pd.DataFrame(
            result_values,
            index=common_clauses,
            columns=region_cols
        )

        # Restore the clause column
        result_df = result_df.reset_index()
        result_df = result_df.rename(columns={"index": clause_col})

        # Output
        output_path = output_folder / pp_path.name
        result_df.to_csv(output_path, index=False, encoding="utf-8-sig")

        print(f"[OK] {pp_path.name} -> {output_path}")
        processed += 1

    except Exception as e:
        print(f"[ERROR] {pp_path.name}: {e}")
        skipped += 1


print("=" * 60)
print(f"Processed: {processed}")
print(f"Skipped  : {skipped}")
print(f"Output folder: {output_folder}")

[INFO] com.brainium.solitairefree.csv:
         California  India  Texas  Vietnam  Bangladesh  Egypt  Germany  \
Clause                                                                  
P1               0      0      0        0           0      0        0   
P2               0      0      0        0           0      0        0   
P3               0      0      0        0           0      0        0   
P4               1      1      1        1           0      0        0   
P5               0      0      0        0           0      0        0   

        Nigeria  Pakistan  Saudi  Turkey  
Clause                                    
P1            0         0      0       0  
P2            0         0      0       0  
P3            0         0      0       0  
P4            0         0      0       0  
P5            0         0      0       0  
[OK] com.brainium.solitairefree.csv -> dataset\AA1_first_batch\AA2_second_100_batch\pp_regulation_matched_output\com.brainium.solitairefree.csv
[IN

### second, construct the region*region matrix for mapped pp, consider pair of regions, which category of clause contribute most, which clause contribute most. another consider is the clause perspective.

In [57]:
import pandas as pd
import numpy as np
from pathlib import Path
from itertools import combinations

# =========================
# Settings
# =========================

input_folder = Path(
    r"dataset\AA1_first_batch\AA2_second_100_batch\pp_regulation_matched_output"
)

output_file = Path(
    r"dataset\AA1_first_batch\AA2_second_100_batch\analysis_pp_output\pp_regulation_matched_pairwise_active_diff_analysis.xlsx"
)

CLAUSE_COL = "Clause"

# If your matrix should contain Clause + 11 regions = 12 columns, enable this check
EXPECTED_TOTAL_COLUMNS = 12

# =========================
# Clause category mapping
# =========================

def get_clause_category(clause):
    clause = str(clause).strip()

    if clause.startswith("CR"):
        return "Controller & Recipient"
    elif clause.startswith("P"):
        return "Personal Data Practices"
    elif clause.startswith("C"):
        return "Consent"
    elif clause.startswith("R"):
        return "Rights"
    else:
        return "Others"

In [44]:
# =========================
# Load one privacy policy matrix
# =========================

def load_policy_matrix(file_path, clause_col=CLAUSE_COL):
    """
    Input CSV format:
        rows = clauses
        columns = Clause + regions
        values = 1 / 0 / np.nan

    Output:
        mat: clause x region matrix
    """

    df = pd.read_csv(file_path)

    # Remove unnamed index columns
    df = df.loc[:, ~df.columns.str.contains(r"^Unnamed")]

    # Optional: skip files whose column number is not expected
    if EXPECTED_TOTAL_COLUMNS is not None and df.shape[1] != EXPECTED_TOTAL_COLUMNS:
        print(f"[SKIP] {file_path.name}: expected {EXPECTED_TOTAL_COLUMNS} columns, got {df.shape[1]}")
        return None

    if clause_col not in df.columns:
        raise ValueError(f"{file_path.name}: Cannot find clause column: {clause_col}")

    mat = df.set_index(clause_col)

    # Remove fully empty region columns
    mat = mat.dropna(axis=1, how="all")

    # Convert values to numeric
    # Any value other than 0 or 1 will be treated as NaN
    mat = mat.apply(pd.to_numeric, errors="coerce")
    mat = mat.where(mat.isin([0, 1]), np.nan)

    return mat


# =========================
# Analyze one app
# =========================

def analyze_one_app(mat, app_name):
    """
    For one app, compute:
    1. region x region pairwise active difference rate matrix
    2. region-pair records
    3. category-level contribution records
    4. clause-level contribution records

    Pairwise active difference rate:
        denominator:
            clauses where both regions have valid 0/1 values
            and at least one of the two regions discloses the clause

        numerator:
            clauses where one region is 1 and the other region is 0

    NaN is not compared.
    """

    regions = list(mat.columns)
    clauses = list(mat.index)

    D = pd.DataFrame(
        np.nan,
        index=regions,
        columns=regions
    )

    # np.fill_diagonal(D.values, 0)
    for r in regions:
        D.loc[r, r] = 0

    pair_records = []
    category_records = []
    clause_records = []

    all_categories = sorted(set(get_clause_category(c) for c in clauses))

    for r1, r2 in combinations(regions, 2):

        v1 = mat[r1]
        v2 = mat[r2]

        # Only compare valid 0/1 pairs
        valid_pair_mask = v1.notna() & v2.notna()

        # Pairwise active clauses:
        # valid 0/1 on both sides, and at least one region discloses
        active_mask = valid_pair_mask & ((v1 == 1) | (v2 == 1))

        active_clauses = mat.index[active_mask]
        denominator = len(active_clauses)

        if denominator == 0:
            diff_count = 0
            diff_rate = np.nan
            diff_clauses = []
        else:
            diff_mask = active_mask & (v1 != v2)
            diff_clauses = list(mat.index[diff_mask])
            diff_count = len(diff_clauses)
            diff_rate = diff_count / denominator

        D.loc[r1, r2] = diff_rate
        D.loc[r2, r1] = diff_rate

        pair_records.append({
            "app": app_name,
            "region_1": r1,
            "region_2": r2,
            "valid_active_clause_count": denominator, # denominator
            "diff_clause_count": diff_count, # numerator
            "pairwise_active_diff_rate": diff_rate # ratio
        })

        # -------------------------
        # Clause-level records
        # -------------------------

        for clause in clauses:
            category = get_clause_category(clause)

            x = mat.loc[clause, r1]
            y = mat.loc[clause, r2]

            both_valid = pd.notna(x) and pd.notna(y)

            # active only if both valid and at least one is 1
            is_active_for_pair = int(
                both_valid and ((x == 1) or (y == 1))
            )

            # diff only if both valid and one is 1, the other is 0
            is_diff = int(
                both_valid and (x != y) and ((x == 1 and y == 0) or (x == 0 and y == 1))
            )

            if denominator == 0:
                contribution = np.nan
            else:
                contribution = is_diff / denominator

            clause_records.append({
                "app": app_name,
                "region_1": r1,
                "region_2": r2,
                "clause": clause,
                "category": category,
                "region_1_value": x,
                "region_2_value": y,
                "both_values_valid_0_1": int(both_valid),
                "is_active_for_pair": is_active_for_pair, # denominator?
                "is_diff": is_diff, # numerator
                "contribution_to_pairwise_active_diff_rate": contribution # ratio
            })

        # -------------------------
        # Category-level records
        # -------------------------

        if denominator == 0:
            for category in all_categories:
                category_records.append({
                    "app": app_name,
                    "region_1": r1,
                    "region_2": r2,
                    "category": category,
                    "category_active_clause_count": 0,
                    "category_diff_clause_count": 0,
                    "category_contribution_to_diff_rate": np.nan,
                    "category_share_among_diff_clauses": np.nan
                })
        else:
            active_temp = []

            for clause in active_clauses:
                category = get_clause_category(clause)
                x = mat.loc[clause, r1]
                y = mat.loc[clause, r2]

                is_diff = int(x != y)

                active_temp.append({
                    "clause": clause,
                    "category": category,
                    "is_diff": is_diff
                })

            active_temp_df = pd.DataFrame(active_temp)

            for category in all_categories:
                sub = active_temp_df[active_temp_df["category"] == category]

                category_active_count = len(sub)
                category_diff_count = int(sub["is_diff"].sum()) if len(sub) > 0 else 0

                category_contribution = category_diff_count / denominator

                if diff_count == 0:
                    category_share = 0
                else:
                    category_share = category_diff_count / diff_count

                category_records.append({
                    "app": app_name,
                    "region_1": r1,
                    "region_2": r2,
                    "category": category,
                    "category_active_clause_count": category_active_count, # denominator
                    "category_diff_clause_count": category_diff_count, # numerator
                    "category_contribution_to_diff_rate": category_contribution, # ratio
                    "category_share_among_diff_clauses": category_share
                })

    return D, pair_records, category_records, clause_records


# =========================
# Average app-level matrices
# =========================

def average_matrices_ignore_nan(app_matrices):
    matrices = list(app_matrices.values())

    if len(matrices) == 0:
        raise ValueError("No valid app matrices found.")

    first = matrices[0]
    stacked = np.stack([m.values for m in matrices], axis=0)

    with np.errstate(all="ignore"):
        avg_values = np.nanmean(stacked, axis=0)

    avg_matrix = pd.DataFrame(
        avg_values,
        index=first.index,
        columns=first.columns
    )

    return avg_matrix


# =========================
# Main
# =========================

app_matrices = {}
all_pair_records = []
all_category_records = []
all_clause_records = []
skipped_files = []

for file_path in input_folder.glob("*.csv"):
    app_name = file_path.stem

    mat = load_policy_matrix(file_path)

    if mat is None:
        skipped_files.append(file_path.name)
        continue

    D, pair_records, category_records, clause_records = analyze_one_app(
        mat,
        app_name
    )

    app_matrices[app_name] = D
    all_pair_records.extend(pair_records)
    all_category_records.extend(category_records)
    all_clause_records.extend(clause_records)

print("Number of valid apps:", len(app_matrices))
print("Number of skipped files:", len(skipped_files))


pair_df = pd.DataFrame(all_pair_records)
category_df = pd.DataFrame(all_category_records)
clause_df = pd.DataFrame(all_clause_records)

path_pair_df = "dataset\\AA1_first_batch\\AA2_second_100_batch\\analysis_pp_output" 
pair_df.to_csv(path_pair_df, index=True)

avg_matrix = average_matrices_ignore_nan(app_matrices)


# =========================
# 1. Region pairs ranking
# =========================

avg_pairwise_df = (
    pair_df
    .groupby(["region_1", "region_2"], as_index=False)
    .agg(
        avg_pairwise_active_diff_rate=("pairwise_active_diff_rate", "mean"),
        avg_valid_active_clause_count=("valid_active_clause_count", "mean"),
        avg_diff_clause_count=("diff_clause_count", "mean"),
        num_apps=("app", "nunique")
    )
    .sort_values("avg_pairwise_active_diff_rate", ascending=False)
)


# =========================
# 2. Category contribution by region pair
# =========================

pair_category_contribution_df = (
    category_df
    .groupby(["region_1", "region_2", "category"], as_index=False)
    .agg(
        avg_category_contribution_to_diff_rate=("category_contribution_to_diff_rate", "mean"),
        avg_category_share_among_diff_clauses=("category_share_among_diff_clauses", "mean"),
        avg_category_active_clause_count=("category_active_clause_count", "mean"),
        avg_category_diff_clause_count=("category_diff_clause_count", "mean"),
        num_apps=("app", "nunique")
    )
    .sort_values(
        ["region_1", "region_2", "avg_category_contribution_to_diff_rate"],
        ascending=[True, True, False]
    )
)

top_category_per_pair_df = (
    pair_category_contribution_df
    .sort_values(
        ["region_1", "region_2", "avg_category_contribution_to_diff_rate"],
        ascending=[True, True, False]
    )
    .groupby(["region_1", "region_2"], as_index=False)
    .head(3)
)


# =========================
# 3. Clause contribution by region pair
# =========================

pair_clause_contribution_df = (
    clause_df
    .groupby(["region_1", "region_2", "clause", "category"], as_index=False)
    .agg(
        avg_clause_contribution_to_diff_rate=("contribution_to_pairwise_active_diff_rate", "mean"),
        diff_frequency_across_apps=("is_diff", "mean"),
        active_frequency_across_apps=("is_active_for_pair", "mean"),
        valid_frequency_across_apps=("both_values_valid_0_1", "mean"),
        num_apps=("app", "nunique")
    )
    .sort_values(
        ["region_1", "region_2", "avg_clause_contribution_to_diff_rate"],
        ascending=[True, True, False]
    )
)

top_clause_per_pair_df = (
    pair_clause_contribution_df
    .sort_values(
        ["region_1", "region_2", "avg_clause_contribution_to_diff_rate"],
        ascending=[True, True, False]
    )
    .groupby(["region_1", "region_2"], as_index=False)
    .head(10)
)


# =========================
# 4. Clause overall ranking
# =========================

clause_overall_ranking_df = (
    clause_df
    .groupby(["clause", "category"], as_index=False)
    .agg(
        avg_contribution_to_pairwise_active_diff_rate=("contribution_to_pairwise_active_diff_rate", "mean"),
        diff_frequency_across_all_app_region_pairs=("is_diff", "mean"),
        active_frequency_across_all_app_region_pairs=("is_active_for_pair", "mean"),
        valid_frequency_across_all_app_region_pairs=("both_values_valid_0_1", "mean"),
        num_records=("app", "count")
    )
    .sort_values(
        "avg_contribution_to_pairwise_active_diff_rate",
        ascending=False
    )
)


# =========================
# 5. Category overall ranking
# =========================

category_overall_ranking_df = (
    category_df
    .groupby("category", as_index=False)
    .agg(
        avg_category_contribution_to_diff_rate=("category_contribution_to_diff_rate", "mean"),
        avg_category_share_among_diff_clauses=("category_share_among_diff_clauses", "mean"),
        avg_category_active_clause_count=("category_active_clause_count", "mean"),
        avg_category_diff_clause_count=("category_diff_clause_count", "mean")
    )
    .sort_values("avg_category_contribution_to_diff_rate", ascending=False)
)


# =========================
# 6. Clause x region-pair difference table
# =========================

clause_region_pair_diff_df = (
    clause_df
    .groupby(["clause", "category", "region_1", "region_2"], as_index=False)
    .agg(
        diff_frequency_across_apps=("is_diff", "mean"),
        active_frequency_across_apps=("is_active_for_pair", "mean"),
        valid_frequency_across_apps=("both_values_valid_0_1", "mean"),
        avg_contribution_to_pairwise_active_diff_rate=("contribution_to_pairwise_active_diff_rate", "mean")
    )
    .sort_values(
        ["clause", "diff_frequency_across_apps"],
        ascending=[True, False]
    )
)


# =========================
# 7. Details for top region pairs
# =========================

top_n_pairs = 10

top_pairs = avg_pairwise_df.head(top_n_pairs)[["region_1", "region_2"]]
top_pair_keys = set(zip(top_pairs["region_1"], top_pairs["region_2"]))

top_pairs_category_detail_df = pair_category_contribution_df[
    pair_category_contribution_df.apply(
        lambda row: (row["region_1"], row["region_2"]) in top_pair_keys,
        axis=1
    )
].sort_values(
    ["region_1", "region_2", "avg_category_contribution_to_diff_rate"],
    ascending=[True, True, False]
)

top_pairs_clause_detail_df = pair_clause_contribution_df[
    pair_clause_contribution_df.apply(
        lambda row: (row["region_1"], row["region_2"]) in top_pair_keys,
        axis=1
    )
].sort_values(
    ["region_1", "region_2", "avg_clause_contribution_to_diff_rate"],
    ascending=[True, True, False]
)


# =========================
# 8. Skipped files table
# =========================

skipped_df = pd.DataFrame({
    "skipped_file": skipped_files
})


# =========================
# Save outputs
# =========================

output_file.parent.mkdir(parents=True, exist_ok=True)

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    # final group-level region x region matrix
    avg_matrix.to_excel(writer, sheet_name="avg_region_matrix")

    # region pair ranking
    avg_pairwise_df.to_excel(writer, sheet_name="region_pair_ranking", index=False)

    # category analysis
    pair_category_contribution_df.to_excel(writer, sheet_name="pair_category_all", index=False)
    top_category_per_pair_df.to_excel(writer, sheet_name="top_categories_per_pair", index=False)
    category_overall_ranking_df.to_excel(writer, sheet_name="category_overall_ranking", index=False)

    # clause analysis
    pair_clause_contribution_df.to_excel(writer, sheet_name="pair_clause_all", index=False)
    top_clause_per_pair_df.to_excel(writer, sheet_name="top_clauses_per_pair", index=False)
    clause_overall_ranking_df.to_excel(writer, sheet_name="clause_overall_ranking", index=False)
    clause_region_pair_diff_df.to_excel(writer, sheet_name="clause_region_pair_diff", index=False)

    # raw records
    pair_df.to_excel(writer, sheet_name="app_pair_raw", index=False)
    category_df.to_excel(writer, sheet_name="app_category_raw", index=False)
    clause_df.to_excel(writer, sheet_name="app_clause_raw", index=False)

    # top-pair details
    top_pairs_category_detail_df.to_excel(writer, sheet_name="top_pairs_category_detail", index=False)
    top_pairs_clause_detail_df.to_excel(writer, sheet_name="top_pairs_clause_detail", index=False)

    # skipped files
    skipped_df.to_excel(writer, sheet_name="skipped_files", index=False)

print(f"Saved analysis to: {output_file}")

print("\nTop region pairs:")
print(avg_pairwise_df.head(10))

print("\nCategory overall ranking:")
print(category_overall_ranking_df)

print("\nClause overall ranking:")
print(clause_overall_ranking_df.head(20))

Number of valid apps: 23
Number of skipped files: 0


PermissionError: [Errno 13] Permission denied: 'dataset\\AA1_first_batch\\AA2_second_100_batch\\analysis_pp_output'

### result

In [58]:
import pandas as pd
import numpy as np
from pathlib import Path
from itertools import combinations

# =========================
# Settings
# =========================

input_folder = Path(
    r"dataset\AA1_first_batch\AA2_second_100_batch\pp_regulation_matched_output"
)

output_file = Path(
    r"dataset\AA1_first_batch\AA2_second_100_batch\analysis_pp_output\pp_regulation_matched_pairwise_active_diff_analysis.xlsx"
)

CLAUSE_COL = "Clause"

# Clause + 11 regions = 12 columns
EXPECTED_TOTAL_COLUMNS = 12


# =========================
# Clause category mapping
# =========================

def get_clause_category(clause):
    clause = str(clause).strip()

    if clause.startswith("CR"):
        return "Controller & Recipient"
    elif clause.startswith("P"):
        return "Personal Data Practices"
    elif clause.startswith("C"):
        return "Consent"
    elif clause.startswith("R"):
        return "Rights"
    else:
        return "Others"


# =========================
# Load one matrix
# =========================

def load_policy_matrix(policy_df, clause_col=CLAUSE_COL):
    """
    Input:
        policy_df:
            rows = clauses
            columns = Clause + regions
            values = 1 / 0 / np.nan

    Output:
        mat:
            rows = clauses
            columns = regions
            values = 1 / 0 / np.nan
    """

    policy_df = policy_df.loc[:, ~policy_df.columns.str.contains(r"^Unnamed")]

    if clause_col not in policy_df.columns:
        raise ValueError(f"Cannot find clause column: {clause_col}")

    mat = policy_df.set_index(clause_col)

    mat = mat.dropna(axis=1, how="all")

    mat = mat.apply(pd.to_numeric, errors="coerce")

    # Only keep 0 and 1. Other values become NaN.
    mat = mat.where(mat.isin([0, 1]), np.nan)

    return mat


# =========================
# Analyze one app
# =========================

def analyze_one_app(mat, app_name):
    """
    Pairwise active difference rate:

    denominator:
        clauses where both regions have valid 0/1 values
        and at least one region discloses the clause.

    numerator:
        clauses where one region is 1 and the other region is 0.

    np.nan is not compared.
    """

    regions = list(mat.columns)
    clauses = list(mat.index)

    D = pd.DataFrame(np.nan, index=regions, columns=regions)

    for r in regions:
        D.loc[r, r] = 0

    pair_records = []
    category_records = []
    clause_records = []

    all_categories = sorted(set(get_clause_category(c) for c in clauses))

    for r1, r2 in combinations(regions, 2):

        v1 = mat[r1]
        v2 = mat[r2]

        valid_pair_mask = v1.notna() & v2.notna()

        # denominator mask
        active_mask = valid_pair_mask & ((v1 == 1) | (v2 == 1))

        # numerator mask
        diff_mask = active_mask & (
            ((v1 == 1) & (v2 == 0)) |
            ((v1 == 0) & (v2 == 1))
        )

        valid_clause_count = int(valid_pair_mask.sum())
        denominator = int(active_mask.sum())
        numerator = int(diff_mask.sum())

        if denominator == 0:
            diff_rate = np.nan
        else:
            diff_rate = numerator / denominator

        D.loc[r1, r2] = diff_rate
        D.loc[r2, r1] = diff_rate

        active_clauses = list(mat.index[active_mask])
        diff_clauses = list(mat.index[diff_mask])

        pair_records.append({
            "app": app_name,
            "region_1": r1,
            "region_2": r2,

            # key values
            "numerator_diff_clause_count": numerator,
            "denominator_valid_active_clause_count": denominator,
            "valid_0_1_clause_count": valid_clause_count,
            "pairwise_active_diff_rate": diff_rate,

            # useful details
            "diff_clauses": "; ".join(diff_clauses),
            "active_clauses": "; ".join(active_clauses)
        })

        # -------------------------
        # Clause-level records
        # -------------------------

        for clause in clauses:
            category = get_clause_category(clause)

            x = mat.loc[clause, r1]
            y = mat.loc[clause, r2]

            both_valid = pd.notna(x) and pd.notna(y)

            is_active = int(
                both_valid and ((x == 1) or (y == 1))
            )

            is_diff = int(
                both_valid and (
                    ((x == 1) and (y == 0)) or
                    ((x == 0) and (y == 1))
                )
            )

            if denominator == 0:
                contribution = np.nan
            else:
                contribution = is_diff / denominator

            clause_records.append({
                "app": app_name,
                "region_1": r1,
                "region_2": r2,
                "clause": clause,
                "category": category,

                "region_1_value": x,
                "region_2_value": y,

                # key values
                "is_valid_0_1_pair": int(both_valid),
                "is_active_for_pair": is_active,
                "is_diff": is_diff,
                "pair_denominator_valid_active_clause_count": denominator,
                "clause_numerator_diff_indicator": is_diff,
                "clause_contribution_to_pairwise_active_diff_rate": contribution
            })

        # -------------------------
        # Category-level records
        # -------------------------

        for category in all_categories:
            category_clauses = [
                c for c in clauses
                if get_clause_category(c) == category
            ]

            category_active_count = 0
            category_diff_count = 0
            category_valid_count = 0

            category_active_clauses = []
            category_diff_clauses = []

            for clause in category_clauses:
                x = mat.loc[clause, r1]
                y = mat.loc[clause, r2]

                both_valid = pd.notna(x) and pd.notna(y)

                if both_valid:
                    category_valid_count += 1

                is_active = both_valid and ((x == 1) or (y == 1))
                is_diff = both_valid and (
                    ((x == 1) and (y == 0)) or
                    ((x == 0) and (y == 1))
                )

                if is_active:
                    category_active_count += 1
                    category_active_clauses.append(str(clause))

                if is_diff:
                    category_diff_count += 1
                    category_diff_clauses.append(str(clause))

            if denominator == 0:
                category_contribution = np.nan
            else:
                # category contribution to the final pairwise active diff rate
                category_contribution = category_diff_count / denominator

            if numerator == 0:
                category_share = np.nan
            else:
                category_share = category_diff_count / numerator

            category_records.append({
                "app": app_name,
                "region_1": r1,
                "region_2": r2,
                "category": category,

                # key values
                "category_numerator_diff_clause_count": category_diff_count,
                "pair_denominator_valid_active_clause_count": denominator,
                "category_contribution_to_diff_rate": category_contribution,

                # category-specific details
                "category_valid_0_1_clause_count": category_valid_count,
                "category_active_clause_count": category_active_count,
                "category_share_among_diff_clauses": category_share,
                "category_diff_clauses": "; ".join(category_diff_clauses),
                "category_active_clauses": "; ".join(category_active_clauses)
            })

    return D, pair_records, category_records, clause_records


# =========================
# Average app-level matrices
# =========================

def average_matrices_ignore_nan(app_matrices):
    matrices = list(app_matrices.values())

    if len(matrices) == 0:
        raise ValueError("No valid app matrices found.")

    first = matrices[0]
    stacked = np.stack([m.values for m in matrices], axis=0)

    avg_values = np.nanmean(stacked, axis=0)

    return pd.DataFrame(
        avg_values,
        index=first.index,
        columns=first.columns
    )


# =========================
# Main
# =========================

app_matrices = {}
all_pair_records = []
all_category_records = []
all_clause_records = []
skipped_files = []

for file_path in input_folder.glob("*.csv"):
    app_name = file_path.stem

    policy_df = pd.read_csv(file_path)
    policy_df = policy_df.loc[:, ~policy_df.columns.str.contains(r"^Unnamed")]

    if EXPECTED_TOTAL_COLUMNS is not None and policy_df.shape[1] != EXPECTED_TOTAL_COLUMNS:
        print(
            f"[SKIP] {app_name}: {policy_df.shape[1]} columns, "
            f"expected {EXPECTED_TOTAL_COLUMNS}."
        )
        skipped_files.append({
            "file": file_path.name,
            "reason": f"{policy_df.shape[1]} columns, expected {EXPECTED_TOTAL_COLUMNS}"
        })
        continue

    mat = load_policy_matrix(policy_df)

    D, pair_records, category_records, clause_records = analyze_one_app(
        mat,
        app_name
    )

    app_matrices[app_name] = D
    all_pair_records.extend(pair_records)
    all_category_records.extend(category_records)
    all_clause_records.extend(clause_records)

print("Number of valid apps:", len(app_matrices))
print("Number of skipped files:", len(skipped_files))


pair_df = pd.DataFrame(all_pair_records)
category_df = pd.DataFrame(all_category_records)
clause_df = pd.DataFrame(all_clause_records)
skipped_df = pd.DataFrame(skipped_files)



avg_matrix = average_matrices_ignore_nan(app_matrices)

path_avg_matrix = "dataset\\AA1_first_batch\\AA2_second_100_batch\\analysis_pp_output\\PP_matched_R_overall_matrix.csv" 
avg_matrix.to_csv(path_avg_matrix, index=True)


# =========================
# 1. Region pair ranking
# =========================

region_pair_ranking_df = (
    pair_df
    .groupby(["region_1", "region_2"], as_index=False)
    .agg(
        # raw numerator / denominator sums
        sum_numerator_diff_clause_count=("numerator_diff_clause_count", "sum"),
        sum_denominator_valid_active_clause_count=("denominator_valid_active_clause_count", "sum"),
        sum_valid_0_1_clause_count=("valid_0_1_clause_count", "sum"),

        # app-level average values
        avg_numerator_diff_clause_count=("numerator_diff_clause_count", "mean"),
        avg_denominator_valid_active_clause_count=("denominator_valid_active_clause_count", "mean"),
        avg_pairwise_active_diff_rate=("pairwise_active_diff_rate", "mean"),

        num_apps=("app", "nunique")
    )
)

region_pair_ranking_df["weighted_pairwise_active_diff_rate"] = (
    region_pair_ranking_df["sum_numerator_diff_clause_count"] /
    region_pair_ranking_df["sum_denominator_valid_active_clause_count"].replace(0, np.nan)
)

region_pair_ranking_df = region_pair_ranking_df.sort_values(
    "avg_pairwise_active_diff_rate",
    ascending=False
)


# =========================
# 2. Category contribution by region pair
# =========================

pair_category_all_df = (
    category_df
    .groupby(["region_1", "region_2", "category"], as_index=False)
    .agg(
        # raw numerator / denominator sums
        sum_category_numerator_diff_clause_count=("category_numerator_diff_clause_count", "sum"),
        sum_pair_denominator_valid_active_clause_count=("pair_denominator_valid_active_clause_count", "sum"),

        # average values
        avg_category_numerator_diff_clause_count=("category_numerator_diff_clause_count", "mean"),
        avg_pair_denominator_valid_active_clause_count=("pair_denominator_valid_active_clause_count", "mean"),
        avg_category_contribution_to_diff_rate=("category_contribution_to_diff_rate", "mean"),

        avg_category_valid_0_1_clause_count=("category_valid_0_1_clause_count", "mean"),
        avg_category_active_clause_count=("category_active_clause_count", "mean"),
        avg_category_share_among_diff_clauses=("category_share_among_diff_clauses", "mean"),

        num_apps=("app", "nunique")
    )
)

pair_category_all_df["weighted_category_contribution_to_diff_rate"] = (
    pair_category_all_df["sum_category_numerator_diff_clause_count"] /
    pair_category_all_df["sum_pair_denominator_valid_active_clause_count"].replace(0, np.nan)
)

pair_category_all_df = pair_category_all_df.sort_values(
    ["region_1", "region_2", "avg_category_contribution_to_diff_rate"],
    ascending=[True, True, False]
)

top_categories_per_pair_df = (
    pair_category_all_df
    .sort_values(
        ["region_1", "region_2", "avg_category_contribution_to_diff_rate"],
        ascending=[True, True, False]
    )
    .groupby(["region_1", "region_2"], as_index=False)
    .head(3)
)


# =========================
# 3. Clause contribution by region pair
# =========================

pair_clause_all_df = (
    clause_df
    .groupby(["region_1", "region_2", "clause", "category"], as_index=False)
    .agg(
        # raw counts
        diff_app_count=("is_diff", "sum"),
        active_app_count=("is_active_for_pair", "sum"),
        valid_app_count=("is_valid_0_1_pair", "sum"),
        total_records=("app", "count"),

        # contribution
        avg_clause_contribution_to_diff_rate=("clause_contribution_to_pairwise_active_diff_rate", "mean"),
        avg_pair_denominator_valid_active_clause_count=("pair_denominator_valid_active_clause_count", "mean"),

        num_apps=("app", "nunique")
    )
)

pair_clause_all_df["diff_frequency_across_apps"] = (
    pair_clause_all_df["diff_app_count"] / pair_clause_all_df["total_records"]
)

pair_clause_all_df["active_frequency_across_apps"] = (
    pair_clause_all_df["active_app_count"] / pair_clause_all_df["total_records"]
)

pair_clause_all_df["valid_frequency_across_apps"] = (
    pair_clause_all_df["valid_app_count"] / pair_clause_all_df["total_records"]
)

pair_clause_all_df["diff_rate_among_active_apps"] = (
    pair_clause_all_df["diff_app_count"] /
    pair_clause_all_df["active_app_count"].replace(0, np.nan)
)

pair_clause_all_df = pair_clause_all_df.sort_values(
    ["region_1", "region_2", "avg_clause_contribution_to_diff_rate"],
    ascending=[True, True, False]
)

top_clauses_per_pair_df = (
    pair_clause_all_df
    .sort_values(
        ["region_1", "region_2", "avg_clause_contribution_to_diff_rate"],
        ascending=[True, True, False]
    )
    .groupby(["region_1", "region_2"], as_index=False)
    .head(10)
)


# =========================
# 4. Clause overall ranking
# =========================

clause_overall_ranking_df = (
    clause_df
    .groupby(["clause", "category"], as_index=False)
    .agg(
        total_diff_count=("is_diff", "sum"),
        total_active_count=("is_active_for_pair", "sum"),
        total_valid_count=("is_valid_0_1_pair", "sum"),
        total_records=("app", "count"),

        avg_contribution_to_pairwise_active_diff_rate=("clause_contribution_to_pairwise_active_diff_rate", "mean"),
        avg_pair_denominator_valid_active_clause_count=("pair_denominator_valid_active_clause_count", "mean")
    )
)

clause_overall_ranking_df["diff_frequency_across_all_app_region_pairs"] = (
    clause_overall_ranking_df["total_diff_count"] /
    clause_overall_ranking_df["total_records"]
)

clause_overall_ranking_df["active_frequency_across_all_app_region_pairs"] = (
    clause_overall_ranking_df["total_active_count"] /
    clause_overall_ranking_df["total_records"]
)

clause_overall_ranking_df["valid_frequency_across_all_app_region_pairs"] = (
    clause_overall_ranking_df["total_valid_count"] /
    clause_overall_ranking_df["total_records"]
)

clause_overall_ranking_df["diff_rate_among_active_records"] = (
    clause_overall_ranking_df["total_diff_count"] /
    clause_overall_ranking_df["total_active_count"].replace(0, np.nan)
)

clause_overall_ranking_df = clause_overall_ranking_df.sort_values(
    "avg_contribution_to_pairwise_active_diff_rate",
    ascending=False
)


# =========================
# 5. Category overall ranking
# =========================

category_overall_ranking_df = (
    category_df
    .groupby("category", as_index=False)
    .agg(
        total_category_numerator_diff_clause_count=("category_numerator_diff_clause_count", "sum"),
        total_pair_denominator_valid_active_clause_count=("pair_denominator_valid_active_clause_count", "sum"),

        avg_category_numerator_diff_clause_count=("category_numerator_diff_clause_count", "mean"),
        avg_pair_denominator_valid_active_clause_count=("pair_denominator_valid_active_clause_count", "mean"),
        avg_category_contribution_to_diff_rate=("category_contribution_to_diff_rate", "mean"),

        avg_category_valid_0_1_clause_count=("category_valid_0_1_clause_count", "mean"),
        avg_category_active_clause_count=("category_active_clause_count", "mean"),
        avg_category_share_among_diff_clauses=("category_share_among_diff_clauses", "mean")
    )
)

category_overall_ranking_df["weighted_category_contribution_to_diff_rate"] = (
    category_overall_ranking_df["total_category_numerator_diff_clause_count"] /
    category_overall_ranking_df["total_pair_denominator_valid_active_clause_count"].replace(0, np.nan)
)

category_overall_ranking_df = category_overall_ranking_df.sort_values(
    "avg_category_contribution_to_diff_rate",
    ascending=False
)


# =========================
# 6. Clause x region-pair diff table
# =========================

clause_region_pair_diff_df = (
    clause_df
    .groupby(["clause", "category", "region_1", "region_2"], as_index=False)
    .agg(
        diff_app_count=("is_diff", "sum"),
        active_app_count=("is_active_for_pair", "sum"),
        valid_app_count=("is_valid_0_1_pair", "sum"),
        total_records=("app", "count"),
        avg_contribution_to_pairwise_active_diff_rate=("clause_contribution_to_pairwise_active_diff_rate", "mean")
    )
)

clause_region_pair_diff_df["diff_frequency_across_apps"] = (
    clause_region_pair_diff_df["diff_app_count"] /
    clause_region_pair_diff_df["total_records"]
)

clause_region_pair_diff_df["active_frequency_across_apps"] = (
    clause_region_pair_diff_df["active_app_count"] /
    clause_region_pair_diff_df["total_records"]
)

clause_region_pair_diff_df["valid_frequency_across_apps"] = (
    clause_region_pair_diff_df["valid_app_count"] /
    clause_region_pair_diff_df["total_records"]
)

clause_region_pair_diff_df["diff_rate_among_active_apps"] = (
    clause_region_pair_diff_df["diff_app_count"] /
    clause_region_pair_diff_df["active_app_count"].replace(0, np.nan)
)

clause_region_pair_diff_df = clause_region_pair_diff_df.sort_values(
    ["clause", "diff_frequency_across_apps"],
    ascending=[True, False]
)


# =========================
# 7. Top region pair details
# =========================

top_n_pairs = 10

top_pairs = region_pair_ranking_df.head(top_n_pairs)[["region_1", "region_2"]]
top_pair_keys = set(zip(top_pairs["region_1"], top_pairs["region_2"]))

top_pairs_category_detail_df = pair_category_all_df[
    pair_category_all_df.apply(
        lambda row: (row["region_1"], row["region_2"]) in top_pair_keys,
        axis=1
    )
].sort_values(
    ["region_1", "region_2", "avg_category_contribution_to_diff_rate"],
    ascending=[True, True, False]
)

top_pairs_clause_detail_df = pair_clause_all_df[
    pair_clause_all_df.apply(
        lambda row: (row["region_1"], row["region_2"]) in top_pair_keys,
        axis=1
    )
].sort_values(
    ["region_1", "region_2", "avg_clause_contribution_to_diff_rate"],
    ascending=[True, True, False]
)


# =========================
# Save
# =========================

output_file.parent.mkdir(parents=True, exist_ok=True)

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    avg_matrix.to_excel(writer, sheet_name="avg_region_matrix")

    region_pair_ranking_df.to_excel(writer, sheet_name="region_pair_ranking", index=False)

    pair_category_all_df.to_excel(writer, sheet_name="pair_category_all", index=False)
    top_categories_per_pair_df.to_excel(writer, sheet_name="top_categories_per_pair", index=False)
    category_overall_ranking_df.to_excel(writer, sheet_name="category_overall_ranking", index=False)

    pair_clause_all_df.to_excel(writer, sheet_name="pair_clause_all", index=False)
    top_clauses_per_pair_df.to_excel(writer, sheet_name="top_clauses_per_pair", index=False)
    clause_overall_ranking_df.to_excel(writer, sheet_name="clause_overall_ranking", index=False)
    clause_region_pair_diff_df.to_excel(writer, sheet_name="clause_region_pair_diff", index=False)

    # raw tables with full numerator / denominator details
    pair_df.to_excel(writer, sheet_name="raw_pair_app_level", index=False)
    category_df.to_excel(writer, sheet_name="raw_category_app_level", index=False)
    clause_df.to_excel(writer, sheet_name="raw_clause_app_level", index=False)

    top_pairs_category_detail_df.to_excel(writer, sheet_name="top_pairs_category_detail", index=False)
    top_pairs_clause_detail_df.to_excel(writer, sheet_name="top_pairs_clause_detail", index=False)

    skipped_df.to_excel(writer, sheet_name="skipped_files", index=False)

print(f"Saved analysis to: {output_file}")

print("\nTop region pairs:")
print(region_pair_ranking_df.head(10))

print("\nCategory overall ranking:")
print(category_overall_ranking_df)

print("\nClause overall ranking:")
print(clause_overall_ranking_df.head(20))

Number of valid apps: 23
Number of skipped files: 0
Saved analysis to: dataset\AA1_first_batch\AA2_second_100_batch\analysis_pp_output\pp_regulation_matched_pairwise_active_diff_analysis.xlsx

Top region pairs:
      region_1    region_2  sum_numerator_diff_clause_count  \
23     Germany      Turkey                                6   
49      Turkey     Vietnam                                3   
48      Turkey       Saudi                                7   
44      Turkey       Egypt                                2   
27       India       Saudi                                4   
7   California       Saudi                                4   
19     Germany       India                                2   
33       Saudi    Pakistan                                3   
32       Saudi  Bangladesh                                3   
22     Germany       Saudi                                3   

    sum_denominator_valid_active_clause_count  sum_valid_0_1_clause_count  \
23                